# Delhivery Last-Mile Delay Intelligence Pipeline
**Capstone Project**  
*Artificial Intelligence and Machine Learning Certification Program*  
**E&ICT Academy, IIT Roorkee & Masai**  
**Author:** Sk Naimuddin  
*GitHub Repository:* [https://github.com/Sk-Naim/delhivery-delay-intelligence-capstone](https://github.com/Sk-Naim/delhivery-delay-intelligence-capstone)

---
### Project Objective
Build an end-to-end, reproducible machine learning pipeline for Delhivery's last-mile logistics operations:
1. **Part A — Data Foundation & EDA**: Clean raw telemetry, engineer domain features, and establish delay incidence evidence.
2. **Part B — Transit Duration Regression**: Predict exact delivery hours with bias-variance and learning rate diagnostics.
3. **Part C — SLA Breach Classification**: Detect delays prior to dispatch with feature selection and asymmetric cost calibration.
4. **Part D — Hub Segmentation & Production Pipeline**: Cluster regional hubs with K-Means and deploy batch scoring logic.


## 0. Pinned Libraries & Environment Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import sklearn
import joblib

print("Environment Versions:")
print(f"Python:       {sys.version.split()[0]}")
print(f"Pandas:       {pd.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Joblib:       {joblib.__version__}")


Environment Versions:
Python:       3.12.3
Pandas:       2.3.2
NumPy:        2.2.6
Scikit-learn: 1.9.1
Joblib:       1.6.0


## 1. Synthetic Data Generation
Run data generator verbatim to create `hubs.csv`, `raw_shipments.csv`, and `new_shipments.csv`.

In [2]:
# Run generate_data.py
import generate_data
# Output verification: 24 hubs, 1224 raw shipments, 48 new shipments


## 2. Inspecting Raw Data Tables

In [3]:
hubs_df = pd.read_csv("data/hubs.csv")
print(f"hubs_df shape: {hubs_df.shape}")
hubs_df.head()


hubs_df shape: (24, 5)


,hub_id,hub_city,region,hub_tier,hub_daily_capacity
0,HUB01,Mumbai,West,1,1500
1,HUB02,Delhi,North,1,1600
2,HUB03,Bengaluru,South,1,1450
3,HUB04,Hyderabad,South,1,1350
4,HUB05,Chennai,South,1,1300


In [4]:
raw_ship_df = pd.read_csv("data/raw_shipments.csv")
print(f"raw_ship_df shape: {raw_ship_df.shape}")
raw_ship_df.head()


raw_ship_df shape: (1224, 17)


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [5]:
# Check duplicates and missing values
duplicate_count = raw_ship_df.duplicated().sum()
duplicate_pct = (duplicate_count / len(raw_ship_df)) * 100
print(f"Exact duplicates: {duplicate_count} ({duplicate_pct:.2f}%)")

print("\nMissing values in raw_ship_df:")
print(raw_ship_df.isna().sum())


Exact duplicates: 24 (1.96%)

Missing values in raw_ship_df:
shipment_id            0
hub_id                 0
service_type           0
payment_mode          18
parcel_weight_kg      25
distance_km           36
declared_value        12
n_stops                0
rain_flag              0
sale_event_flag        0
dispatch_time          0
label_print_batch      0
scanner_build_no       0
tape_roll_no           0
sticker_serial_mod     0
shelf_slot_no          0
delivery_hours         0
dtype: int64


## 3. Part A — Sequential Data Cleaning & Feature Engineering

In [6]:
from src.shipment_prep import PROMISED_HOURS, FEATURES, NOISE_COLUMNS, clean_shipments, build_features

print("PROMISED_HOURS mapping:")
print(PROMISED_HOURS)

print(f"\nFEATURES ({len(FEATURES)} inputs):")
print(FEATURES)

print(f"\nNOISE_COLUMNS ({len(NOISE_COLUMNS)} synthetic noise features):")
print(NOISE_COLUMNS)


PROMISED_HOURS mapping:
{'Express': 36, 'Standard': 60, 'Economy': 96}

FEATURES (17 inputs):
['promised_hours', 'distance_km', 'parcel_weight_kg', 'log_declared_value', 'n_stops', 'rain_flag', 'sale_event_flag', 'is_cod', 'is_weekend', 'dispatch_hour', 'hub_tier', 'hub_daily_capacity', 'label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no']

NOISE_COLUMNS (5 synthetic noise features):
['label_print_batch', 'scanner_build_no', 'tape_roll_no', 'sticker_serial_mod', 'shelf_slot_no']


In [7]:
# Step-by-step cleaning execution
clean_df, medians = clean_shipments(raw_ship_df)
print(f"Cleaned shape (after dedup and cleaning): {clean_df.shape}")
print(f"Computed imputation medians: {medians}")
clean_df.head()


Cleaned shape (after dedup and cleaning): (1200, 17)
Computed imputation medians: {'distance_km': 314.95, 'parcel_weight_kg': 1.83, 'declared_value_inr': 1121.5}


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours,declared_value_inr
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,4594,25,167,0.650,13,34.79,1290.0
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,4066,31,169,0.045,243,58.56,1030.0
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,3090,25,170,0.915,80,76.56,441.0
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,1796,21,338,0.380,175,41.18,3122.0
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,4231,34,215,0.824,162,47.32,276.0


In [8]:
# Feature engineering & hub merge (validate='many_to_one')
modelling_table = build_features(clean_df, hubs_df)

# Define delay target
modelling_table["is_delayed"] = (
    modelling_table["delivery_hours"] > modelling_table["promised_hours"]
).astype(int)

# Persist modelling table
os.makedirs("data", exist_ok=True)
modelling_table.to_csv("data/modelling_table.csv", index=False)

print(f"Modelling table shape: {modelling_table.shape}")
print(f"Zero missing values verification: {modelling_table[FEATURES + ['delivery_hours', 'is_delayed']].isna().sum().sum()}")
modelling_table.head()


Modelling table shape: (1200, 27)
Zero missing values verification: 0


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,hub_city,region,hub_tier,hub_daily_capacity,promised_hours,is_cod,is_weekend,dispatch_hour,log_declared_value,is_delayed
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,Lucknow,North,2,900,36,1,1,19,7.162397,0
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,Lucknow,North,2,900,60,1,1,7,6.937314,0
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,Bhubaneswar,East,3,390,96,0,1,12,6.089045,0
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,Bengaluru,South,1,1450,60,0,0,17,8.046229,0
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,Coimbatore,South,2,740,60,0,1,17,5.620401,0


## 4. Part A — Exploratory Data Analysis & Delay Evidence

In [9]:
# Overall delay metrics
total_shipments = len(modelling_table)
delayed_count = int(modelling_table["is_delayed"].sum())
delay_rate = delayed_count / total_shipments

print(f"Total Shipments: {total_shipments}")
print(f"Delayed Parcels: {delayed_count}")
print(f"Network Delay Rate: {delay_rate:.4f} ({delay_rate*100:.2f}%)")


Total Shipments: 1200
Delayed Parcels: 278
Network Delay Rate: 0.2317 (23.17%)


In [10]:
# Group delay by Service Type
service_summary = (
    modelling_table
    .groupby("service_type")
    .agg(
        shipments=("shipment_id", "count"),
        delayed=("is_delayed", "sum"),
        mean_hours=("delivery_hours", "mean")
    )
    .assign(delay_rate=lambda df: (df["delayed"] / df["shipments"]).round(4))
    .reset_index()
)
service_summary


,service_type,shipments,delayed,mean_hours,delay_rate
0,Economy,315,27,82.093111,0.0857
1,Express,243,119,36.805638,0.4897
2,Standard,642,132,52.256916,0.2056


In [11]:
# Group delay by Hub Tier
tier_summary = (
    modelling_table
    .groupby("hub_tier")
    .agg(
        shipments=("shipment_id", "count"),
        delayed=("is_delayed", "sum"),
        mean_hours=("delivery_hours", "mean")
    )
    .assign(delay_rate=lambda df: (df["delayed"] / df["shipments"]).round(4))
    .reset_index()
)
tier_summary


,hub_tier,shipments,delayed,mean_hours,delay_rate
0,1,400,16,48.566375,0.040
1,2,400,86,58.819400,0.215
2,3,400,176,63.494325,0.440


In [12]:
# Distance Bands Analysis
modelling_table["distance_band"] = pd.cut(
    modelling_table["distance_km"],
    bins=[0, 200, 400, 600, 2000],
    labels=["0-200", "201-400", "401-600", "601+"],
    right=True
)

dist_summary = (
    modelling_table
    .groupby("distance_band", observed=True)
    .agg(
        shipments=("shipment_id", "count"),
        delayed=("is_delayed", "sum"),
        mean_hours=("delivery_hours", "mean")
    )
    .assign(delay_rate=lambda df: (df["delayed"] / df["shipments"]).round(4))
    .reset_index()
)
dist_summary


,distance_band,shipments,delayed,mean_hours,delay_rate
0,0-200,274,6,46.755876,0.0219
1,201-400,563,77,55.965950,0.1368
2,401-600,275,120,64.266000,0.4364
3,601+,88,75,72.260795,0.8523


In [13]:
# Run Part A standalone pipeline script
import part_a_clean_eda
part_a_clean_eda.run_part_a()


Part A complete. Outputs generated in data/ and outputs/.


## 5. Part B — Delivery Hours Regression & Diagnostics

In [14]:
import part_b_regression
part_b_regression.run_part_b()


Part B complete. Best regression model: linear_regression


In [15]:
# View Part B regression benchmarks
metrics_b = pd.read_csv("outputs/part_b_metrics.csv")
print("Regression Benchmark Metrics:")
metrics_b


Regression Benchmark Metrics:


,model,split,MAE,MSE,RMSE,R2
0,mean_baseline,val,15.7952,355.5204,18.8552,-0.0212
1,linear_regression,val,3.9222,23.9235,4.8912,0.9313
2,decision_tree,val,5.0971,39.4338,6.2796,0.8867
3,random_forest,val,4.2541,30.0132,5.4784,0.9138
4,gradient_boosting,val,3.9166,24.1722,4.9165,0.9306
5,linear_regression,test,3.5671,20.4624,4.5235,0.9493


In [16]:
# Tree Depth Bias-Variance Analysis
tree_depth_b = pd.read_csv("outputs/part_b_tree_depth.csv")
print("Decision Tree Depth Analysis:")
tree_depth_b


Decision Tree Depth Analysis:


,max_depth,train_rmse,val_rmse
0,1,11.6334,12.1324
1,2,9.3687,10.4114
2,3,7.5295,8.7033
3,4,6.0999,7.3896
4,6,4.7114,6.2796
5,8,3.3360,6.8385
6,12,1.3613,7.4173
7,unlimited,0.0000,7.7890


In [17]:
# Gradient Boosting Learning Rate Grid
gb_lr_b = pd.read_csv("outputs/part_b_gb_learning_rate.csv")
print("Gradient Boosting Learning Rate Grid:")
gb_lr_b


Gradient Boosting Learning Rate Grid:


,learning_rate,val_rmse,val_r2
0,0.01,10.0018,0.7127
1,0.10,4.9165,0.9306
2,1.00,7.3701,0.8440


## 6. Part C — Delay Classification & Cost Calibration

In [18]:
import part_c_classification
part_c_classification.run_part_c()


Part C complete. Chosen configuration: default_forest_same_7_features, Alert threshold: 0.5, Test F1: 0.7209


In [19]:
# Candidate Model Performance (Validation Set)
models_c = pd.read_csv("outputs/part_c_models.csv")
print("Part C Validation Benchmarks:")
models_c


Part C Validation Benchmarks:


,model,TN,FP,FN,TP,accuracy,precision,recall,f1
0,blind_all_on_time,138,0,42,0,0.7667,0.0000,0.0000,0.0000
1,logistic_regression,128,10,7,35,0.9056,0.7778,0.8333,0.8046
2,decision_tree_depth_6,128,10,11,31,0.8833,0.7561,0.7381,0.7470
3,random_forest_default,132,6,10,32,0.9111,0.8421,0.7619,0.8000


In [20]:
# Feature Selection (SelectKBest f_classif)
feat_scores_c = pd.read_csv("outputs/part_c_feature_scores.csv")
print("Top 7 Selected Features vs Rejected Noise Columns:")
feat_scores_c.sort_values("f_rank")


Top 7 Selected Features vs Rejected Noise Columns:


,feature,f_score,f_rank,rf_importance,is_noise
0,distance_km,389.2131,1,0.2707,False
1,hub_tier,155.3483,2,0.0399,False
2,hub_daily_capacity,144.1347,3,0.0816,False
3,n_stops,125.4798,4,0.0704,False
4,promised_hours,84.4177,5,0.0870,False
5,rain_flag,53.0321,6,0.0449,False
6,sale_event_flag,36.7281,7,0.0333,False
7,label_print_batch,4.5847,8,0.0462,True
8,is_cod,3.8949,9,0.0104,False
9,is_weekend,2.6939,10,0.0087,False


In [21]:
# Decision Threshold Sweep
thresholds_c = pd.read_csv("outputs/part_c_thresholds.csv")
print("Probability Threshold Sweep (0.1 to 0.9):")
thresholds_c


Probability Threshold Sweep (0.1 to 0.9):


,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1
0,0.1,94,44,1,41,0.7500,0.4824,0.9762,0.6457
1,0.2,109,29,3,39,0.8222,0.5735,0.9286,0.7091
2,0.3,121,17,3,39,0.8889,0.6964,0.9286,0.7959
3,0.4,125,13,5,37,0.9000,0.7400,0.8810,0.8043
4,0.5,129,9,7,35,0.9111,0.7955,0.8333,0.8140
5,0.6,134,4,10,32,0.9222,0.8889,0.7619,0.8205
6,0.7,134,4,15,27,0.8944,0.8710,0.6429,0.7397
7,0.8,134,4,23,19,0.8500,0.8261,0.4524,0.5846
8,0.9,136,2,30,12,0.8222,0.8571,0.2857,0.4286


In [22]:
# Final Test Set Evaluation (180 Unseen Shipments)
final_test_c = pd.read_csv("outputs/part_c_final_test.csv")
print("Final Blind Test Performance:")
final_test_c


Final Blind Test Performance:


,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1
0,0.5,125,14,10,31,0.8667,0.6889,0.7561,0.7209


## 7. Part D — Hub Clustering & Operational Engine

In [23]:
import part_d_segments
part_d_segments.run_part_d_segments()


Part D segmentation complete. Hub segments and separation evidence saved.


In [24]:
# K-Means Separation Diagnostics (K=2 to 5)
separation_d = pd.read_csv("outputs/part_d_separation.csv")
print("K-Means Separation Across-over-Within Ratios:")
separation_d


K-Means Separation Across-over-Within Ratios:


,K,across_over_within,cluster_sizes
0,2,2.8940,15-9
1,3,5.3928,9-8-7
2,4,6.3047,8-7-5-4
3,5,6.8866,8-7-4-3-2


In [25]:
# Segment Operational Profiles
segment_profile_d = pd.read_csv("outputs/segment_profile.csv")
print("Operational Segment Summary:")
segment_profile_d


Operational Segment Summary:


,segment,hubs,mean_delivery_hours,delay_rate,mean_distance_km,mean_n_stops
0,Standard,7,47.2396,0.0257,188.9296,55.3686
1,Watch,8,56.8137,0.1550,314.1762,70.7175
2,Escalate,9,64.6504,0.4600,470.4757,84.3444


In [26]:
# Tier vs Segment Mismatch Hubs
mismatch_d = pd.read_csv("outputs/tier_vs_segment_mismatch.csv")
print("Mismatch Hubs (Operational Performance vs Administrative Tier):")
mismatch_d


Mismatch Hubs (Operational Performance vs Administrative Tier):


,hub_id,segment,hub_city,hub_tier
0,HUB07,Watch,Kolkata,1
1,HUB13,Escalate,Nagpur,2
2,HUB15,Escalate,Kochi,2
3,HUB22,Watch,Dehradun,3


In [27]:
# Package Model Artifacts & Verify Reload
import package_model
package_model.run_package_model()


Packaging complete: 180 test rows scored. Max abs prob diff: 0.0e+00. Threshold: 0.5


In [28]:
# Operational Batch Scoring on New Dispatches
import predict_delay
predict_delay.predict_delays()


Scoring complete for 48 new shipments. Flagged: 8 (Escalate: 6, Call: 2, None: 40)


In [29]:
# Preview Operational Dispatch Actions
actions_df = pd.read_csv("outputs/new_shipment_actions.csv")
print(f"Scored {len(actions_df)} new dispatches:")
print("\nAction Breakdown:")
print(actions_df["action"].value_counts())
print("\nSample Actions:")
actions_df.head(10)


Scored 48 new dispatches:

Action Breakdown:
action
No action                  40
Escalate to hub manager     6
Proactive customer call     2
Name: count, dtype: int64

Sample Actions:


,shipment_id,hub_id,segment,delay_probability,flagged,action
0,SHP90001,HUB22,Watch,0.09,0,No action
1,SHP90002,HUB24,Escalate,0.74,1,Escalate to hub manager
2,SHP90003,HUB18,Escalate,0.88,1,Escalate to hub manager
3,SHP90004,HUB12,Watch,0.00,0,No action
4,SHP90005,HUB22,Watch,0.73,1,Proactive customer call
5,SHP90006,HUB12,Watch,0.06,0,No action
6,SHP90007,HUB09,Watch,0.53,1,Proactive customer call
7,SHP90008,HUB01,Standard,0.01,0,No action
8,SHP90009,HUB09,Watch,0.08,0,No action
9,SHP90010,HUB13,Escalate,0.47,0,No action


## 8. Summary & Acceptance Verification
All pipeline stages executed end-to-end matching 100% of the Capstone specifications.